# Coach 长期画像辅助对比 Demo

这份 notebook 只做一件事：
用 `wrong_binder` 里的一道现成例题，直观对比 `coach_agent` 在“没加长期画像”和“加了长期画像”时的差异。

这份 notebook 的设计原则：

1. 不保存任何结果文件
2. 固定同一道题、同一条学生回答、同一个当前错因
3. 只改变 `student_memory_profile`
4. 先看本地可解释差异，再决定要不要真实调用模型看首轮回复差异


In [1]:
import difflib
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from coach_agent import (
    CoachSession,
    ErrorType,
    FoundryCoachAgent,
    analyze_student_reply,
    apply_student_memory_bias,
    build_reply_analysis_fallback,
    build_student_memory_context,
    build_student_memory_strategy_note,
    build_turn_prompt,
    diagnose_environment,
    get_coach_strategy,
    infer_completed_from_reply,
)
from review_scheduler import build_review_schedule


def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2))


def show_block(title: str, text: str) -> None:
    print(f'\n{title}')
    print('-' * len(title))
    print(text.strip() if str(text).strip() else '（空）')


def show_prompt_diff(left: str, right: str, *, left_name: str, right_name: str) -> None:
    diff = difflib.unified_diff(
        left.splitlines(),
        right.splitlines(),
        fromfile=left_name,
        tofile=right_name,
        lineterm='',
    )
    diff_text = '\n'.join(diff).strip()
    print(diff_text if diff_text else '两份 prompt 完全一致。')


## 1. 读取一条 `wrong_binder` 例题

这里直接使用现成 fixture：`binder_case_sequence_shift.json`。

为了把注意力集中在 `coach_agent` 的长期画像辅助上，下面会把“当前题当前轮的错因”固定成 `missing_strategy`，这样前后对比时只有长期画像在变化。

In [2]:
CASE_PATH = PROJECT_ROOT / 'harness' / 'fixtures' / 'wrong_binder' / 'binder_case_sequence_shift.json'
CASE_RECORD = json.loads(CASE_PATH.read_text(encoding='utf-8'))
QUESTION = CASE_RECORD['question_payload']

PROBLEM_TEXT = QUESTION['stem']
STUDENT_REPLY = QUESTION['student_answer']
REFERENCE_ANSWER = QUESTION['solution_text']
STUDENT_PROFILE = '学生能从递推式入手，但经常不会自己提出“先构造一个新数列”这种中间目标。'
CURRENT_ERROR_TYPE = ErrorType.MISSING_STRATEGY

show_json(
    {
        'case_id': CASE_RECORD['case_id'],
        'description': CASE_RECORD['description'],
        'problem_text': PROBLEM_TEXT,
        'student_reply': STUDENT_REPLY,
        'fixed_error_type_for_compare': CURRENT_ERROR_TYPE.value,
    }
)


{
  "case_id": "binder_case_sequence_shift",
  "description": "递推构造等比数列证明题",
  "problem_text": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
  "student_reply": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。",
  "fixed_error_type_for_compare": "missing_strategy"
}


## 2. 先假定一个长期画像

这个画像是人为构造的，不依赖前面生成过的 `student_memory_profile` 文件。

这里故意让它强调 `strategy_first`，这样你更容易看出：

- 没加长期画像时，coach 只根据当前题当前轮来定策略
- 加了长期画像后，coach 会轻度偏向“先说解题路线 / 中间目标”


In [3]:
FAKE_MEMORY_PROFILE = {
    'record_id': 'student_memory.demo_compare',
    'student_id': 'demo_compare',
    'personalization_summary': {
        'memory_stage': 'forming_pattern',
        'dominant_error_type': 'missing_strategy',
        'dominant_error_signal_strength': 'established',
        'recommended_teaching_mode': 'strategy_first',
        'recommended_review_mode': 'mixed',
        'top_recurrent_nodes': [
            {
                'node_id': 'math.数列与不等式.数列.基础概念.数列递推公式解读',
                'title': '数列递推公式解读',
            },
            {
                'node_id': 'math.数列与不等式.数列.递推数列转化方法.构造等比数列',
                'title': '构造等比数列',
            },
        ],
        'notes': [
            '这个学生往往知道局部变形，但不会先说出“要把递推式改造成 b_(n+1)=qb_n”这样的路线。',
            '讲题时要优先提醒中间目标，而不是直接堆局部代数步骤。',
        ],
    },
}

show_json(FAKE_MEMORY_PROFILE['personalization_summary'])


{
  "memory_stage": "forming_pattern",
  "dominant_error_type": "missing_strategy",
  "dominant_error_signal_strength": "established",
  "recommended_teaching_mode": "strategy_first",
  "recommended_review_mode": "mixed",
  "top_recurrent_nodes": [
    {
      "node_id": "math.数列与不等式.数列.基础概念.数列递推公式解读",
      "title": "数列递推公式解读"
    },
    {
      "node_id": "math.数列与不等式.数列.递推数列转化方法.构造等比数列",
      "title": "构造等比数列"
    }
  ],
  "notes": [
    "这个学生往往知道局部变形，但不会先说出“要把递推式改造成 b_(n+1)=qb_n”这样的路线。",
    "讲题时要优先提醒中间目标，而不是直接堆局部代数步骤。"
  ]
}


## 3. 构造“无长期画像 / 有长期画像”两套 session

这里先不真实调用模型，只看本地可解释层：

- `reply_analysis`
- `base_strategy`
- `memory_bias`
- `turn_prompt`

这样更容易看清，到底是哪一层发生了变化。

In [4]:
heuristic_quality = analyze_student_reply(STUDENT_REPLY)
reply_analysis = build_reply_analysis_fallback(
    heuristic_quality,
    '本 notebook 为了稳定对比，不调用 reply analyzer，直接使用本地 fallback。',
    completed=infer_completed_from_reply(STUDENT_REPLY),
)

base_strategy = get_coach_strategy(
    CURRENT_ERROR_TYPE,
    reply_analysis.quality,
    turn_index=0,
    total_turns=4,
    understands=reply_analysis.understands,
    completed=reply_analysis.completed,
)

session_plain = CoachSession(
    problem_text=PROBLEM_TEXT,
    error_type=CURRENT_ERROR_TYPE,
    student_profile=STUDENT_PROFILE,
    initial_student_reply=STUDENT_REPLY,
)

session_memory = CoachSession(
    problem_text=PROBLEM_TEXT,
    error_type=CURRENT_ERROR_TYPE,
    student_profile=STUDENT_PROFILE,
    student_memory_profile=FAKE_MEMORY_PROFILE,
    student_memory_context=build_student_memory_context(FAKE_MEMORY_PROFILE),
    initial_student_reply=STUDENT_REPLY,
)

strategy_plain = apply_student_memory_bias(base_strategy, session_plain)
strategy_memory = apply_student_memory_bias(base_strategy, session_memory)

prompt_plain = build_turn_prompt(
    session=session_plain,
    student_reply=STUDENT_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_plain,
)

prompt_memory = build_turn_prompt(
    session=session_memory,
    student_reply=STUDENT_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_memory,
)

show_json(
    {
        'reply_analysis': reply_analysis.as_dict(),
        'base_strategy': base_strategy.as_dict(),
        'strategy_without_memory': strategy_plain.as_dict(),
        'strategy_with_memory': strategy_memory.as_dict(),
    }
)


{
  "reply_analysis": {
    "quality": "empty",
    "understands": false,
    "completed": false,
    "reason": "本 notebook 为了稳定对比，不调用 reply analyzer，直接使用本地 fallback。",
    "source": "fallback_heuristic"
  },
  "base_strategy": {
    "mode": "socratic_light",
    "trap": "学生知道局部知识，但不会先确定中间目标。",
    "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。"
  },
  "strategy_without_memory": {
    "mode": "socratic_light",
    "trap": "学生知道局部知识，但不会先确定中间目标。",
    "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。"
  },
  "strategy_with_memory": {
    "mode": "socratic_light",
    "trap": "学生知道局部知识，但不会先确定中间目标。",
    "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。结合长期画像：优先先点出中间目标和解题路线，不要只给局部结论。"
  }
}


## 4. 直观看差异

这里按三个层次看：

1. 长期画像上下文本身是什么
2. 本轮策略话术有没有被轻微改写
3. 最终发给模型的 turn prompt 差在哪里


In [5]:
show_block('一、无长期画像：student_memory_context', session_plain.student_memory_context)
show_block('二、有长期画像：student_memory_context', session_memory.student_memory_context)

show_block('三、无长期画像：本轮策略话术', strategy_plain.as_prompt_block())
show_block('四、有长期画像：本轮策略话术', strategy_memory.as_prompt_block())

memory_note = build_student_memory_strategy_note(base_strategy, session_memory)
show_block('五、长期画像额外加上的策略提示', memory_note)



一、无长期画像：student_memory_context
------------------------------
（空）

二、有长期画像：student_memory_context
------------------------------
记忆阶段：已形成一定模式
长期主错因：missing_strategy
长期信号强度：established
长期教学偏好：strategy_first
长期复习偏好：mixed
当前最常卡叶子：数列递推公式解读
辅助备注：这个学生往往知道局部变形，但不会先说出“要把递推式改造成 b_(n+1)=qb_n”这样的路线。 讲题时要优先提醒中间目标，而不是直接堆局部代数步骤。

三、无长期画像：本轮策略话术
--------------
教学模式：socratic_light
学生可能卡点：学生知道局部知识，但不会先确定中间目标。
本轮策略话术：学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。

四、有长期画像：本轮策略话术
--------------
教学模式：socratic_light
学生可能卡点：学生知道局部知识，但不会先确定中间目标。
本轮策略话术：学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。结合长期画像：优先先点出中间目标和解题路线，不要只给局部结论。

五、长期画像额外加上的策略提示
---------------
结合长期画像：优先先点出中间目标和解题路线，不要只给局部结论。


In [6]:
show_block('六、无长期画像：完整 turn prompt', prompt_plain)
show_block('七、有长期画像：完整 turn prompt', prompt_memory)



六、无长期画像：完整 turn prompt
----------------------
【题目】
已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。

【学生画像】
学生能从递推式入手，但经常不会自己提出“先构造一个新数列”这种中间目标。

【学生错误类型】
missing_strategy

【学生最新回答】
我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。



【学生理解度分析工具结果】
回答质量：empty
是否理解：否
是否答完整：否
判定来源：fallback_heuristic
判定理由：本 notebook 为了稳定对比，不调用 reply analyzer，直接使用本地 fallback。
教学模式：socratic_light
本轮策略话术：学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。

【回复要求】
这是第一轮 coach 回复，必须把开场、判断和下一步引导说完整。
额外规则：语气亲切，适合中小学生，用词通俗易懂，避免使用专业术语。
请只输出 coach 对学生说的话，不要输出分析过程、字段名或 JSON。
最多 3 句话，必须完整回答，结尾用句号、问号或感叹号。
如果“是否答完整”为“否”，你这一轮只能补全当前题，不能给变式练习。

七、有长期画像：完整 turn prompt
----------------------
【题目】
已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。

【学生画像】
学生能从递推式入手，但经常不会自己提出“先构造一个新数列”这种中间目标。

【学生错误类型】
missing_strategy

【学生最新回答】
我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。


【学生长期记忆辅助信息】
记忆阶段：已形成一定模式
长期主错因：missing_strategy
长期信号强度：established
长期教学偏好：strategy_first
长期复习偏好：mixed
当前最常卡叶子：数列递推公式解读
辅助备注：这个学生往往知道局部变形，但不会先说出“要把

In [7]:
print('八、turn prompt 的差异 diff')
print('==========================')
show_prompt_diff(
    prompt_plain,
    prompt_memory,
    left_name='without_memory',
    right_name='with_memory',
)


八、turn prompt 的差异 diff
--- without_memory
+++ with_memory
@@ -11,6 +11,15 @@
 我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。
 
 
+【学生长期记忆辅助信息】
+记忆阶段：已形成一定模式
+长期主错因：missing_strategy
+长期信号强度：established
+长期教学偏好：strategy_first
+长期复习偏好：mixed
+当前最常卡叶子：数列递推公式解读
+辅助备注：这个学生往往知道局部变形，但不会先说出“要把递推式改造成 b_(n+1)=qb_n”这样的路线。 讲题时要优先提醒中间目标，而不是直接堆局部代数步骤。
+使用规则：这只是辅助偏好，不要覆盖当前题的本轮诊断；如果冲突，以当前题当前轮的真实卡点为主。
 
 【学生理解度分析工具结果】
 回答质量：empty
@@ -19,7 +28,7 @@
 判定来源：fallback_heuristic
 判定理由：本 notebook 为了稳定对比，不调用 reply analyzer，直接使用本地 fallback。
 教学模式：socratic_light
-本轮策略话术：学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。
+本轮策略话术：学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。结合长期画像：优先先点出中间目标和解题路线，不要只给局部结论。
 
 【回复要求】
 这是第一轮 coach 回复，必须把开场、判断和下一步引导说完整。


## 5. 可选：真实调用模型看首轮 coach 回复差异

默认关闭，避免你每次打开 notebook 都立刻打 Azure 请求。

如果你想看真实差异，把 `RUN_REAL_MODEL` 改成 `True` 即可。

注意：这一部分仍然不会保存结果文件，只会在 notebook 里打印。

In [9]:
RUN_REAL_MODEL = True
MODEL_DEPLOYMENT = 'gpt-4o-mini'
USE_DEFAULT_CREDENTIAL = False

print('coach_environment:')
show_json(diagnose_environment())

if RUN_REAL_MODEL:
    agent = FoundryCoachAgent(
        model_deployment=MODEL_DEPLOYMENT,
        use_default_credential=USE_DEFAULT_CREDENTIAL,
        use_ai_reply_analyzer=False,
    )

    real_plain_session = agent.create_session(
        problem_text=PROBLEM_TEXT,
        error_type=CURRENT_ERROR_TYPE,
        student_profile=STUDENT_PROFILE,
        student_memory_profile=None,
        initial_student_reply=STUDENT_REPLY,
    )
    response_plain = agent.start(real_plain_session, student_reply=STUDENT_REPLY)

    real_memory_session = agent.create_session(
        problem_text=PROBLEM_TEXT,
        error_type=CURRENT_ERROR_TYPE,
        student_profile=STUDENT_PROFILE,
        student_memory_profile=FAKE_MEMORY_PROFILE,
        initial_student_reply=STUDENT_REPLY,
    )
    response_memory = agent.start(real_memory_session, student_reply=STUDENT_REPLY)

    show_block('九、无长期画像：首轮 coach 回复', response_plain.content)
    show_block('十、有长期画像：首轮 coach 回复', response_memory.content)

    show_json(
        {
            'without_memory_strategy': response_plain.strategy.as_dict(),
            'with_memory_strategy': response_memory.strategy.as_dict(),
            'without_memory_reply_analysis': response_plain.reply_analysis.as_dict(),
            'with_memory_reply_analysis': response_memory.reply_analysis.as_dict(),
        }
    )
else:
    print('当前未调用模型。把 RUN_REAL_MODEL 改成 True 后，可以直接对比首轮 coach 回复。')


coach_environment:
{
  "coach_agent_version": "2026-06-16-structured-output-check",
  "az_path": "/opt/homebrew/bin/az",
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}

九、无长期画像：首轮 coach 回复
-------------------
你好！从递推式可以先算出前几项，看看 a_n 和 ½ 的关系会发现 a_n+½ 好像每一步都乘了同一个数。你可以尝试把 a_n 减去 ‑½ （即考虑 b_n = a_n + ½ ），把递推式写成关于 b_n 的形式，看看会得到什么样的关系？这样一步一步来，你会发现 b_n 恰好是等比数列哦。

十、有长期画像：首轮 coach 回复
-------------------
你好，我看到你已经找到了递推式，但还没有想到把它改写成乘一个固定数的形式。我们可以先把每一项加上 ½，看看新数列 b_n = a_n+½ 的递推关系会变成什么样。你试着把 b_{n+1}=a_{n+1}+½ 用 b_n 表示一下，看看会得到什么系数？
{
  "without_memory_strategy": {
    "mode": "socratic_light",
    "trap": "学生知道局部知识，但不会先确定中间目标。",
    "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。"
  },
  "with_memory_strategy": {
    "mode": "socratic_light",
    "trap": "学生知道局部知识，但不会先确定中间目标。",
    "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。结合长期画像：优先先点出中间目标和解题路线，不要只给局部结论。"
  

## 6. 对比 `review_scheduler` 的长期画像微调

这一部分不再看 `coach_agent`，而是看复习排序。

目标是验证：

1. 不加长期画像时，系统按原始遗忘 / 到期 / 错误压力去排
2. 加了长期画像后，和该学生长期薄弱点相关的知识点、题目会被轻度前移
3. 依旧只打印结果，不保存任何文件


In [ ]:
REVIEW_STATE_PATH = PROJECT_ROOT / 'scratch' / 'student_annotation_merged_review_state_v2.json'
REVIEW_STATE = json.loads(REVIEW_STATE_PATH.read_text(encoding='utf-8'))

REVIEW_MEMORY_PROFILE = {
    'record_id': 'student_memory.review_compare_demo',
    'student_id': 'demo_compare',
    'personalization_summary': {
        'memory_stage': 'forming_pattern',
        'dominant_error_type': 'missing_strategy',
        'dominant_error_signal_strength': 'established',
        'recommended_teaching_mode': 'strategy_first',
        'recommended_review_mode': 'mixed',
        'top_recurrent_nodes': [
            {
                'node_id': 'math.数列与不等式.数列.基础概念.数列递推公式解读',
                'title': '数列递推公式解读',
            },
            {
                'node_id': 'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
                'title': '标准型递推公式转化',
            },
        ],
        'notes': [
            '学生在递推题里经常知道从 a_(n+1) 与 a_n 入手，但不知道该把递推式变形成什么目标结构。',
            '复习时可轻度优先推送递推解读与递推转化类知识点，并把对应错题往前放。',
        ],
    },
    'node_memories': [
        {
            'node_id': 'math.数列与不等式.数列.基础概念.数列递推公式解读',
            'observed_wrong_count': 2,
            'review_wrong_count': 1,
            'practice_request_count': 1,
            'signal_strength': 'established',
            'recommended_intervention': 'show_strategy_first',
        },
        {
            'node_id': 'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化',
            'observed_wrong_count': 1,
            'review_wrong_count': 1,
            'practice_request_count': 1,
            'signal_strength': 'tentative',
            'recommended_intervention': 'show_strategy_first',
        },
    ],
    'question_memories': [
        {
            'question_id': 'wq_demo_001',
            'wrong_count': 2,
            'review_count': 1,
            'signal_strength': 'established',
            'last_result': 'wrong',
        }
    ],
}

show_json(
    {
        'review_state_path': str(REVIEW_STATE_PATH),
        'knowledge_point_state_count': len(REVIEW_STATE.get('knowledge_point_states', [])),
        'example_question_state_count': len(REVIEW_STATE.get('example_question_states', [])),
        'review_memory_summary': REVIEW_MEMORY_PROFILE['personalization_summary'],
    }
)


In [ ]:
def find_rank_and_item(items, key, value):
    for index, item in enumerate(items, 1):
        if item.get(key) == value:
            return index, item
    return None, None


def summarize_priority_item(item, key_name):
    if item is None:
        return None
    return {
        key_name: item.get(key_name),
        'priority_score': item.get('priority_score'),
        'memory_priority_boost': item.get('memory_priority_boost'),
        'memory_priority_reason': item.get('memory_priority_reason'),
        'state': item.get('state'),
        'next_review_at': item.get('next_review_at'),
    }


plain_schedule = build_review_schedule(
    REVIEW_STATE,
    node_limit=50,
    question_limit=50,
    student_memory_profile=None,
).as_dict()

memory_schedule = build_review_schedule(
    REVIEW_STATE,
    node_limit=50,
    question_limit=50,
    student_memory_profile=REVIEW_MEMORY_PROFILE,
).as_dict()

TARGET_NODE_A = 'math.数列与不等式.数列.基础概念.数列递推公式解读'
TARGET_NODE_B = 'math.数列与不等式.数列.递推数列转化方法.标准型递推公式转化'
TARGET_QUESTION = 'wq_demo_001'

plain_rank_a, plain_item_a = find_rank_and_item(plain_schedule['node_priorities'], 'node_id', TARGET_NODE_A)
memory_rank_a, memory_item_a = find_rank_and_item(memory_schedule['node_priorities'], 'node_id', TARGET_NODE_A)

plain_rank_b, plain_item_b = find_rank_and_item(plain_schedule['node_priorities'], 'node_id', TARGET_NODE_B)
memory_rank_b, memory_item_b = find_rank_and_item(memory_schedule['node_priorities'], 'node_id', TARGET_NODE_B)

plain_rank_q, plain_item_q = find_rank_and_item(plain_schedule['question_priorities'], 'question_id', TARGET_QUESTION)
memory_rank_q, memory_item_q = find_rank_and_item(memory_schedule['question_priorities'], 'question_id', TARGET_QUESTION)

show_json(
    {
        'target_node_compare': [
            {
                'node_id': TARGET_NODE_A,
                'without_memory_rank': plain_rank_a,
                'with_memory_rank': memory_rank_a,
                'without_memory': summarize_priority_item(plain_item_a, 'node_id'),
                'with_memory': summarize_priority_item(memory_item_a, 'node_id'),
            },
            {
                'node_id': TARGET_NODE_B,
                'without_memory_rank': plain_rank_b,
                'with_memory_rank': memory_rank_b,
                'without_memory': summarize_priority_item(plain_item_b, 'node_id'),
                'with_memory': summarize_priority_item(memory_item_b, 'node_id'),
            },
        ],
        'target_question_compare': {
            'question_id': TARGET_QUESTION,
            'without_memory_rank': plain_rank_q,
            'with_memory_rank': memory_rank_q,
            'without_memory': summarize_priority_item(plain_item_q, 'question_id'),
            'with_memory': summarize_priority_item(memory_item_q, 'question_id'),
        },
    }
)


In [ ]:
show_json(
    {
        'without_memory_top_5_nodes': [
            {
                'rank': index,
                'node_id': item['node_id'],
                'priority_score': item['priority_score'],
                'memory_priority_boost': item['memory_priority_boost'],
            }
            for index, item in enumerate(plain_schedule['node_priorities'][:5], 1)
        ],
        'with_memory_top_5_nodes': [
            {
                'rank': index,
                'node_id': item['node_id'],
                'priority_score': item['priority_score'],
                'memory_priority_boost': item['memory_priority_boost'],
            }
            for index, item in enumerate(memory_schedule['node_priorities'][:5], 1)
        ],
        'without_memory_top_5_questions': [
            {
                'rank': index,
                'question_id': item['question_id'],
                'priority_score': item['priority_score'],
                'memory_priority_boost': item['memory_priority_boost'],
            }
            for index, item in enumerate(plain_schedule['question_priorities'][:5], 1)
        ],
        'with_memory_top_5_questions': [
            {
                'rank': index,
                'question_id': item['question_id'],
                'priority_score': item['priority_score'],
                'memory_priority_boost': item['memory_priority_boost'],
            }
            for index, item in enumerate(memory_schedule['question_priorities'][:5], 1)
        ],
    }
)


你重点看三件事：

1. `数列递推公式解读` 是否从原本普通位置被轻微前移
2. `标准型递推公式转化` 是否因为长期画像而进入更靠前的位置
3. `wq_demo_001` 这道题是否因为历史上反复错而被推到更靠前

如果这三件事都发生了，就说明 `review_scheduler` 的长期画像微调已经真实进入排序链路，而且是“轻推”而不是“重写排序”。